# 02 — Fixture catalog (offline gold)

The sandbox ships a tiny, attributed catalog so every runner can execute
without Hub, OpenRouter, or a GPU. This notebook walks the catalog the
same way [`datasets.py`](../src/mailroom_sandbox/datasets.py) does.

Sources:

- [`data/fixtures/manifest.csv`](../data/fixtures/manifest.csv)
- [`data/fixtures/ATTRIBUTION.md`](../data/fixtures/ATTRIBUTION.md)
- [`data/fixtures/hf/docclass_mini.jsonl`](../data/fixtures/hf/docclass_mini.jsonl)
- [`data/fixtures/legalbench/contract_qa.jsonl`](../data/fixtures/legalbench/contract_qa.jsonl)
- [`data/fixtures/agents/*.jsonl`](../data/fixtures/agents)
- [`data/fixtures/intake/hello.pdf`](../data/fixtures/intake/hello.pdf)


In [ ]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    """Walk up from cwd (hostile kernels start in notebooks/)."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "reports").is_dir():
            return cand
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

from _lib import bootstrap, isolate_outputs

ROOT = bootstrap(ROOT)
OUT = isolate_outputs(ROOT)
print("repo root :", ROOT)
print("sys.path[0]:", sys.path[0])
print("notebook outputs ->", OUT)


## Manifest rows and class mix


In [ ]:
from collections import Counter
from mailroom_sandbox.datasets import (
    load_manifest,
    fixture_file,
    parse_expected_fields,
    dataset_fingerprint,
    load_hf_fixtures,
    load_legalbench_fixtures,
    load_agent_fixtures,
    agent_fixture_path,
)
from mailroom_sandbox.paths import fixtures_dir

rows = load_manifest()
print("n manifest rows:", len(rows))
print("fingerprint    :", dataset_fingerprint(rows))
print("classes        :", dict(Counter(r["expected_doc_class"] for r in rows)))
print("stages         :", dict(Counter(r.get("expected_stage") for r in rows)))
print()
print(f"{'id':24s} {'class':20s} {'stage':10s} file")
for row in rows:
    path = fixture_file(row)
    print(f"{row['id']:24s} {row['expected_doc_class']:20s} {row.get('expected_stage','?'):10s} {path.name} exists={path.is_file()}")


## One contract fixture + typed expected fields


In [ ]:
msa = next(r for r in rows if r["id"] == "contract_msa")
text = fixture_file(msa).read_text(encoding="utf-8")
fields = parse_expected_fields(msa)
print("id:", msa["id"])
print("path:", fixture_file(msa))
print("--- first 400 chars ---")
print(text[:400])
print("--- expected_fields (gold for extraction evals) ---")
print(fields)
print()
amb = next(r for r in rows if r["id"] == "ambiguous_01")
print("ambiguous_01 expected class:", amb["expected_doc_class"], "stage:", amb.get("expected_stage"))
print("(the mock LLM assigns confidence 0.40 to this id so routing evals have a REVIEW case)")


## HF mini-slice, LegalBench Yes/No, per-agent gold


In [ ]:
hf = load_hf_fixtures()
lb = load_legalbench_fixtures()
print("HF mini rows:", len(hf), "keys sample:", sorted(hf[0]) if hf else None)
print("HF classes :", sorted({r.get("expected_hf_class") or r.get("doc_type") for r in hf}))
print("LegalBench n:", len(lb), "task answers:", [r.get("answer") for r in lb])
print()
for agent in ("intake", "pdf_transcriber", "image_extractor", "judge", "arbiter", "boss"):
    gold = load_agent_fixtures(agent)
    print(f"agents/{agent}.jsonl -> {len(gold):2d} rows  ({agent_fixture_path(agent).name})")


## Provenance and honest gaps


In [ ]:
attr = (fixtures_dir() / "ATTRIBUTION.md").read_text(encoding="utf-8")
print(attr[:900])
print()
png = fixtures_dir() / "intake" / "hello.png"
pdf = fixtures_dir() / "intake" / "hello.pdf"
print("hello.pdf exists:", pdf.is_file(), "size", pdf.stat().st_size if pdf.is_file() else 0)
print("hello.png exists:", png.is_file(), "— image_extractor live path wants a PNG; mock uses agents/image_extractor.jsonl")
print()
print("HONEST GAP: this catalog is synthetic / attributed snippets, not CUAD or")
print("docclass-merged scale. `sandbox datasets pull` is the Hub path (network).")
